# `build_ledger.ipynb` — CSV-only ledger prototype

Pulls from live Corral + `notebooks/out/corral_transition_table.csv`, writes six CSVs under `data/` and two under `views/`, runs the accounting-identity validator, and compares 2023 SFRUU totals to the public report.

Kernel: `arcgispro-py3`. Read-only against Corral.

## 1. Imports + paths + Corral engine

In [ ]:
from __future__ import annotations

import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from sqlalchemy import text

NB_DIR = Path.cwd()
REPO_ROOT = NB_DIR.parent if NB_DIR.name == "ledger_prototype" else NB_DIR
PROTO = NB_DIR if NB_DIR.name == "ledger_prototype" else NB_DIR / "ledger_prototype"
DATA = PROTO / "data"
VIEWS = PROTO / "views"
DATA.mkdir(parents=True, exist_ok=True)
VIEWS.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(REPO_ROOT / "erd"))
from db_corral import get_engine  # noqa: E402

engine = get_engine()
LOADED_AT = datetime.now(timezone.utc).isoformat()
print(f"DATA:  {DATA}")
print(f"VIEWS: {VIEWS}")
print(f"LOADED_AT: {LOADED_AT}")

## 2. `conversion_ratio.csv` — hardcoded from 2013 Regional Plan

12 rows: `600 CFA = 2 TAU = 2 SFRUU = 3 MFRUU` in every direction. Reference lookup, not ledger data.

In [ ]:
CONV_PAIRS = [
    ("CFA",   600, "TAU",   2),
    ("CFA",   600, "SFRUU", 2),
    ("CFA",   600, "MFRUU", 3),
    ("TAU",   2,   "SFRUU", 2),
    ("TAU",   2,   "MFRUU", 3),
    ("SFRUU", 2,   "MFRUU", 3),
]
rows = []
for f_c, f_q, t_c, t_q in CONV_PAIRS:
    rows.append({"FromCommodity": f_c, "FromQty": f_q, "ToCommodity": t_c, "ToQty": t_q})
    rows.append({"FromCommodity": t_c, "FromQty": t_q, "ToCommodity": f_c, "ToQty": f_q})
conv = pd.DataFrame(rows)
conv["EffectiveFrom"] = "2013-01-01"
conv["EffectiveTo"] = None
conv["Source"] = "2013 Regional Plan"
conv.to_csv(DATA / "conversion_ratio.csv", index=False)
print(f"conversion_ratio.csv: {len(conv)} rows")
conv

## 3. `account.csv` — chart of accounts

Three parcel-keyed buckets per `(Jurisdiction × Commodity)` + one pool-keyed bucket per `CommodityPool` (classified as `BonusUnits` or `UnusedCapacity` by name heuristic).

`MaxCapacity` is left null until the analyst confirms the authoritative source per commodity (open Q3 in the email draft).

In [ ]:
with engine.connect() as conn:
    jurs = pd.read_sql(text(
        "SELECT JurisdictionID, ResidentialAllocationAbbreviation AS Abbrev "
        "FROM dbo.Jurisdiction"
    ), conn)
    commodities = pd.read_sql(text(
        "SELECT CommodityID, CommodityShortName, CommodityDisplayName FROM dbo.Commodity"
    ), conn)
    pools = pd.read_sql(text(
        "SELECT cp.CommodityPoolID, cp.CommodityPoolName, cp.CommodityID, "
        "       c.CommodityShortName "
        "FROM dbo.CommodityPool cp "
        "JOIN dbo.Commodity c ON c.CommodityID = cp.CommodityID"
    ), conn)
print(f"jurs={len(jurs)}  commodities={len(commodities)}  pools={len(pools)}")

parcel_bucket_rows = []
for _, j in jurs.iterrows():
    for _, c in commodities.iterrows():
        for b in ("Existing", "Banked", "Allocated"):
            parcel_bucket_rows.append({
                "AccountScope": "jurisdiction",
                "CommodityPoolID": None,
                "JurisdictionID": int(j.JurisdictionID),
                "JurisdictionAbbrev": j.Abbrev,
                "CommodityID": int(c.CommodityID),
                "CommodityShortName": c.CommodityShortName,
                "BucketType": b,
                "MaxCapacity": None,
            })

def classify_pool(name: str) -> str:
    n = str(name).lower()
    if "bonus" in n:
        return "BonusUnits"
    return "UnusedCapacity"

pool_rows = []
for _, p in pools.iterrows():
    pool_rows.append({
        "AccountScope": "pool",
        "CommodityPoolID": int(p.CommodityPoolID),
        "JurisdictionID": None,
        "JurisdictionAbbrev": None,
        "CommodityID": int(p.CommodityID),
        "CommodityShortName": p.CommodityShortName,
        "BucketType": classify_pool(p.CommodityPoolName),
        "MaxCapacity": None,
    })

accounts = pd.DataFrame(parcel_bucket_rows + pool_rows)
accounts.insert(0, "AccountID", range(1, len(accounts) + 1))
accounts.to_csv(DATA / "account.csv", index=False)
print(f"account.csv: {len(accounts):,} rows")
print(accounts["BucketType"].value_counts())

## 4. Ledger branch 1 — `corral_tdr`

Pulls `dbo.TdrTransaction` + all child tables, fans each transaction out into 1–2 ledger rows based on `MovementType`. `EventID = TdrTransactionID` (so CONV/CONVTRF/TRF paired rows share an EventID).

Simplifications for v0 of the prototype:
- Shorezone (`SHORE`) excluded via `TransactionType` filter.
- LandBank acquisitions/transfers map into `BucketType='Banked'` (LandBank as pseudo-bucket). Mechanically correct for the accounting identity; refine semantics later.
- ECM retirement is a single debit row (commodity leaves the system — `SUM(Quantity) GROUP BY EventID` is negative by design for this MovementType only).

In [ ]:
with engine.connect() as conn:
    tdr = pd.read_sql(text(
        "SELECT tt.TdrTransactionID, tt.ApprovalDate, tt.CommodityID AS FromCommodityID, "
        "       c.CommodityShortName AS FromCommodityShort, "
        "       tt.TransactionTypeID, tty.TransactionTypeAbbreviation AS Movement "
        "FROM dbo.TdrTransaction tt "
        "JOIN dbo.TransactionType tty ON tty.TransactionTypeID = tt.TransactionTypeID "
        "LEFT JOIN dbo.Commodity c    ON c.CommodityID         = tt.CommodityID "
        "WHERE tty.TransactionTypeAbbreviation <> 'SHORE'"
    ), conn)
    parcels = pd.read_sql(text(
        "SELECT ParcelID, ParcelNumber, JurisdictionID FROM dbo.Parcel"
    ), conn)
    alloc = pd.read_sql(text(
        "SELECT TdrTransactionID, SendingAllocationPoolID, ReceivingParcelID, AllocatedQuantity "
        "FROM dbo.TdrTransactionAllocation"
    ), conn)
    trf = pd.read_sql(text(
        "SELECT TdrTransactionID, SendingParcelID, SendingQuantity, ReceivingParcelID, ReceivingQuantity "
        "FROM dbo.TdrTransactionTransfer"
    ), conn)
    conv_t = pd.read_sql(text(
        "SELECT ttc.TdrTransactionID, ttc.ParcelID, ttc.CommodityConvertedToID, "
        "       ttc.SendingQuantity, ttc.ReceivingQuantity, "
        "       c2.CommodityShortName AS ToCommodityShort "
        "FROM dbo.TdrTransactionConversion ttc "
        "JOIN dbo.Commodity c2 ON c2.CommodityID = ttc.CommodityConvertedToID"
    ), conn)
    conv_trf = pd.read_sql(text(
        "SELECT ttct.TdrTransactionID, ttct.SendingParcelID, ttct.ReceivingParcelID, "
        "       ttct.SendingQuantity, ttct.ReceivingQuantity, "
        "       ttct.CommodityConvertedToID, c2.CommodityShortName AS ToCommodityShort "
        "FROM dbo.TdrTransactionConversionWithTransfer ttct "
        "JOIN dbo.Commodity c2 ON c2.CommodityID = ttct.CommodityConvertedToID"
    ), conn)
    ecm = pd.read_sql(text(
        "SELECT TdrTransactionID, SendingParcelID, RetirementQuantity FROM dbo.TdrTransactionECMRetirement"
    ), conn)
    lba = pd.read_sql(text(
        "SELECT TdrTransactionID, SendingParcelID, SendingQuantity, LandBankID "
        "FROM dbo.TdrTransactionLandBankAcquisition"
    ), conn)
    lbt = pd.read_sql(text(
        "SELECT TdrTransactionID, SendingLandBankID, ReceivingParcelID, SendingQuantity "
        "FROM dbo.TdrTransactionLandBankTransfer"
    ), conn)

print("Row counts:", {
    "tdr": len(tdr), "alloc": len(alloc), "trf": len(trf),
    "conv": len(conv_t), "conv_trf": len(conv_trf),
    "ecm": len(ecm), "lba": len(lba), "lbt": len(lbt),
})

apn_of = dict(zip(parcels.ParcelID, parcels.ParcelNumber))
jur_id_of = dict(zip(parcels.ParcelID, parcels.JurisdictionID))
jur_abbrev_of_id = dict(zip(jurs.JurisdictionID, jurs.Abbrev))
jur_abbrev = lambda pid: jur_abbrev_of_id.get(jur_id_of.get(pid))

alloc_by_tt = {r.TdrTransactionID: r for r in alloc.itertuples(index=False)}
trf_by_tt   = {r.TdrTransactionID: r for r in trf.itertuples(index=False)}
conv_by_tt  = {r.TdrTransactionID: r for r in conv_t.itertuples(index=False)}
ctrf_by_tt  = {r.TdrTransactionID: r for r in conv_trf.itertuples(index=False)}
ecm_by_tt   = {r.TdrTransactionID: r for r in ecm.itertuples(index=False)}
lba_by_tt   = {r.TdrTransactionID: r for r in lba.itertuples(index=False)}
lbt_by_tt   = {r.TdrTransactionID: r for r in lbt.itertuples(index=False)}

In [ ]:
LEDGER_COLS = [
    "EntryID", "EventID", "EntryDate", "Year",
    "JurisdictionAbbrev", "CommodityPoolID", "CommodityShortName",
    "BucketType", "Quantity", "MovementType",
    "ParcelNumber", "SourceSystem", "SourceID", "Notes",
]

def entry(event_id, date, jur, pool, commodity, bucket, qty, movement, parcel_id=None, note=""):
    return {
        "EntryID": None,
        "EventID": int(event_id),
        "EntryDate": date,
        "Year": pd.Timestamp(date).year if pd.notna(date) else None,
        "JurisdictionAbbrev": jur,
        "CommodityPoolID": int(pool) if pool is not None and pd.notna(pool) else None,
        "CommodityShortName": commodity,
        "BucketType": bucket,
        "Quantity": float(qty),
        "MovementType": movement,
        "ParcelNumber": apn_of.get(parcel_id) if parcel_id is not None else None,
        "SourceSystem": "corral_tdr",
        "SourceID": f"corral_tdr:{int(event_id)}",
        "Notes": note,
    }

ledger_rows = []
skipped = {}

for tt_row in tdr.itertuples(index=False):
    m = tt_row.Movement
    ttid = tt_row.TdrTransactionID
    date = tt_row.ApprovalDate
    from_commodity = tt_row.FromCommodityShort

    if m == "ALLOC":
        c = alloc_by_tt.get(ttid)
        if c is None:
            skipped[m] = skipped.get(m, 0) + 1; continue
        ledger_rows.append(entry(ttid, date, None, c.SendingAllocationPoolID, from_commodity,
                                 "UnusedCapacity", -c.AllocatedQuantity, m))
        ledger_rows.append(entry(ttid, date, jur_abbrev(c.ReceivingParcelID), None, from_commodity,
                                 "Allocated", +c.AllocatedQuantity, m, c.ReceivingParcelID))
    elif m == "ALLOCASSGN":
        c = alloc_by_tt.get(ttid)
        if c is None:
            skipped[m] = skipped.get(m, 0) + 1; continue
        jur = jur_abbrev(c.ReceivingParcelID)
        ledger_rows.append(entry(ttid, date, jur, None, from_commodity, "Allocated",
                                 -c.AllocatedQuantity, m, c.ReceivingParcelID))
        ledger_rows.append(entry(ttid, date, jur, None, from_commodity, "Existing",
                                 +c.AllocatedQuantity, m, c.ReceivingParcelID))
    elif m == "TRF":
        c = trf_by_tt.get(ttid)
        if c is None:
            skipped[m] = skipped.get(m, 0) + 1; continue
        ledger_rows.append(entry(ttid, date, jur_abbrev(c.SendingParcelID), None, from_commodity,
                                 "Existing", -c.SendingQuantity, m, c.SendingParcelID))
        ledger_rows.append(entry(ttid, date, jur_abbrev(c.ReceivingParcelID), None, from_commodity,
                                 "Existing", +c.ReceivingQuantity, m, c.ReceivingParcelID))
    elif m == "CONV":
        c = conv_by_tt.get(ttid)
        if c is None:
            skipped[m] = skipped.get(m, 0) + 1; continue
        jur = jur_abbrev(c.ParcelID)
        ledger_rows.append(entry(ttid, date, jur, None, from_commodity, "Existing",
                                 -c.SendingQuantity, m, c.ParcelID, "convert FROM"))
        ledger_rows.append(entry(ttid, date, jur, None, c.ToCommodityShort, "Existing",
                                 +c.ReceivingQuantity, m, c.ParcelID, "convert TO"))
    elif m == "CONVTRF":
        c = ctrf_by_tt.get(ttid)
        if c is None:
            skipped[m] = skipped.get(m, 0) + 1; continue
        ledger_rows.append(entry(ttid, date, jur_abbrev(c.SendingParcelID), None, from_commodity, "Existing",
                                 -c.SendingQuantity, m, c.SendingParcelID, "convert+transfer FROM"))
        ledger_rows.append(entry(ttid, date, jur_abbrev(c.ReceivingParcelID), None, c.ToCommodityShort, "Existing",
                                 +c.ReceivingQuantity, m, c.ReceivingParcelID, "convert+transfer TO"))
    elif m == "ECM":
        c = ecm_by_tt.get(ttid)
        if c is None:
            skipped[m] = skipped.get(m, 0) + 1; continue
        ledger_rows.append(entry(ttid, date, jur_abbrev(c.SendingParcelID), None, from_commodity,
                                 "Existing", -c.RetirementQuantity, m, c.SendingParcelID, "retired (out of system)"))
    elif m == "LBA":
        c = lba_by_tt.get(ttid)
        if c is None:
            skipped[m] = skipped.get(m, 0) + 1; continue
        ledger_rows.append(entry(ttid, date, jur_abbrev(c.SendingParcelID), None, from_commodity,
                                 "Existing", -c.SendingQuantity, m, c.SendingParcelID))
        ledger_rows.append(entry(ttid, date, None, None, from_commodity,
                                 "Banked", +c.SendingQuantity, m, None, f"LandBankID={c.LandBankID}"))
    elif m == "LBT":
        c = lbt_by_tt.get(ttid)
        if c is None:
            skipped[m] = skipped.get(m, 0) + 1; continue
        ledger_rows.append(entry(ttid, date, None, None, from_commodity,
                                 "Banked", -c.SendingQuantity, m, None, f"LandBankID={c.SendingLandBankID}"))
        ledger_rows.append(entry(ttid, date, jur_abbrev(c.ReceivingParcelID), None, from_commodity,
                                 "Existing", +c.SendingQuantity, m, c.ReceivingParcelID))
    else:
        skipped[m] = skipped.get(m, 0) + 1

corral_tdr_rows = pd.DataFrame(ledger_rows, columns=LEDGER_COLS)
print(f"corral_tdr rows: {len(corral_tdr_rows):,}")
print("By MovementType:")
print(corral_tdr_rows["MovementType"].value_counts())
if skipped:
    print("Skipped:", skipped)

## 5. Ledger branch 2 — `corral_banking`

`dbo.ParcelPermitBankedDevelopmentRight` doesn't have a row in `TdrTransaction`; banking is inferred from the banked-right record + the permit's `FinalInspectionDate` (proxy for banking date — flagged as open question Q5 in `erd/target_schema.md`).

Paired Existing → Banked rows per banked right.

In [ ]:
with engine.connect() as conn:
    banking = pd.read_sql(text(
        "SELECT ppbdr.ParcelPermitBankedDevelopmentRightID AS BankingID, "
        "       ppbdr.ParcelPermitID, ppbdr.LandCapabilityTypeID, ppbdr.Quantity, "
        "       pp.ParcelID, pp.FinalInspectionDate, "
        "       lct.CommodityID, c.CommodityShortName "
        "FROM dbo.ParcelPermitBankedDevelopmentRight ppbdr "
        "JOIN dbo.ParcelPermit pp           ON pp.ParcelPermitID      = ppbdr.ParcelPermitID "
        "JOIN dbo.LandCapabilityType lct    ON lct.LandCapabilityTypeID = ppbdr.LandCapabilityTypeID "
        "JOIN dbo.Commodity c               ON c.CommodityID           = lct.CommodityID"
    ), conn)
print(f"banking rows: {len(banking):,}")

event_offset = int(corral_tdr_rows["EventID"].max() or 0) + 1

banking_rows = []
for i, b in enumerate(banking.itertuples(index=False)):
    eid = event_offset + i
    date = b.FinalInspectionDate
    jur = jur_abbrev(b.ParcelID)
    apn = apn_of.get(b.ParcelID)
    year = pd.Timestamp(date).year if pd.notna(date) else None
    banking_rows.append({
        "EntryID": None, "EventID": eid, "EntryDate": date, "Year": year,
        "JurisdictionAbbrev": jur, "CommodityPoolID": None,
        "CommodityShortName": b.CommodityShortName,
        "BucketType": "Existing", "Quantity": -float(b.Quantity), "MovementType": "Banking",
        "ParcelNumber": apn, "SourceSystem": "corral_banking",
        "SourceID": f"corral_banking:{int(b.BankingID)}", "Notes": "",
    })
    banking_rows.append({
        "EntryID": None, "EventID": eid, "EntryDate": date, "Year": year,
        "JurisdictionAbbrev": jur, "CommodityPoolID": None,
        "CommodityShortName": b.CommodityShortName,
        "BucketType": "Banked", "Quantity": +float(b.Quantity), "MovementType": "Banking",
        "ParcelNumber": apn, "SourceSystem": "corral_banking",
        "SourceID": f"corral_banking:{int(b.BankingID)}", "Notes": "",
    })

corral_banking_rows = pd.DataFrame(banking_rows, columns=LEDGER_COLS)
print(f"corral_banking ledger rows: {len(corral_banking_rows):,}")

## 6. Ledger branch 3 — `manual`

Empty placeholder for future QA corrections / manual adjustments.

In [ ]:
manual_rows = pd.DataFrame(columns=LEDGER_COLS)
print(f"manual rows: {len(manual_rows)} (placeholder)")

## 7. Concatenate + write `ledger_entry.csv`

In [ ]:
ledger = pd.concat([corral_tdr_rows, corral_banking_rows, manual_rows], ignore_index=True)
ledger = ledger.sort_values(["EntryDate", "EventID"], kind="stable").reset_index(drop=True)
ledger["EntryID"] = range(1, len(ledger) + 1)
ledger.to_csv(DATA / "ledger_entry.csv", index=False)
print(f"ledger_entry.csv: {len(ledger):,} rows")
print("By SourceSystem:")
print(ledger["SourceSystem"].value_counts())
print("\nBy MovementType:")
print(ledger["MovementType"].value_counts())
print("\nBy BucketType:")
print(ledger["BucketType"].value_counts())

## 8. `ledger_entry_annotation.csv`

Inner-join `notebooks/out/corral_transition_table.csv` (from the earlier diff work) to `ledger_entry.csv` on `TdrTransactionID` (parsed back out of `SourceID`). Orphans go to `_orphan_annotations.csv` for review.

In [ ]:
transition_csv = REPO_ROOT / "notebooks" / "out" / "corral_transition_table.csv"
if not transition_csv.exists():
    print(f"!! {transition_csv} not found — run notebooks/02_build_transition_table.ipynb first.")
    annotations = pd.DataFrame()
else:
    tt_df = pd.read_csv(transition_csv)
    print(f"transition table rows: {len(tt_df):,}")

    ledger_tdr = ledger[ledger["SourceSystem"] == "corral_tdr"].copy()
    ledger_tdr["TdrTransactionID"] = ledger_tdr["SourceID"].str.replace("corral_tdr:", "", regex=False).astype(int)

    target = ledger_tdr[ledger_tdr["MovementType"] == "ALLOCASSGN"].drop_duplicates("TdrTransactionID")

    joined = tt_df.merge(target[["TdrTransactionID", "EntryID"]], on="TdrTransactionID", how="left")
    matched = joined[joined["EntryID"].notna()].copy()
    orphans = joined[joined["EntryID"].isna()].copy()

    annotation_cols = [
        "EntryID", "AllocationNumber", "TransactionCreatedDate", "TransactionAcknowledgedDate",
        "DevelopmentType", "DetailedDevelopmentType", "TrpaMouProjectNumber",
        "YearBuilt", "PmYearBuilt", "SupplementalNotes",
        "SourceFile", "SourceRowNumber", "LoadedAt",
    ]
    annotations = matched.reindex(columns=annotation_cols)
    annotations.insert(0, "AnnotationID", range(1, len(annotations) + 1))
    annotations["EntryID"] = annotations["EntryID"].astype("Int64")
    annotations.to_csv(DATA / "ledger_entry_annotation.csv", index=False)
    if len(orphans) > 0:
        orphans.to_csv(DATA / "_orphan_annotations.csv", index=False)

    print(f"ledger_entry_annotation.csv: {len(annotations):,} rows")
    print(f"orphans (no matching ledger entry): {len(orphans):,}")
    if len(tt_df):
        print(f"linkage rate: {len(annotations)/len(tt_df):.1%}")

## 9. `v_cumulative_accounting.csv`

Running balances by `(Year, JurisdictionAbbrev, CommodityShortName, BucketType)`. One row per combination per year; `Quantity` is the year-end balance (cumulative sum through that year).

Pool-keyed entries (no JurisdictionAbbrev) roll up into a synthetic `TRPA` row.

In [ ]:
led = ledger.copy()
led["JurisdictionAbbrev"] = led["JurisdictionAbbrev"].fillna("TRPA")
led = led.dropna(subset=["Year"])
led["Year"] = led["Year"].astype(int)

per_year = (
    led.groupby(["Year", "JurisdictionAbbrev", "CommodityShortName", "BucketType"], as_index=False)["Quantity"].sum()
)
per_year = per_year.sort_values(["JurisdictionAbbrev", "CommodityShortName", "BucketType", "Year"])
per_year["RunningBalance"] = (
    per_year.groupby(["JurisdictionAbbrev", "CommodityShortName", "BucketType"])["Quantity"].cumsum()
)
snapshot = per_year.pivot_table(
    index=["Year", "JurisdictionAbbrev", "CommodityShortName"],
    columns="BucketType",
    values="RunningBalance",
    aggfunc="last",
).reset_index()
snapshot.columns.name = None
snapshot.to_csv(VIEWS / "v_cumulative_accounting.csv", index=False)
print(f"v_cumulative_accounting.csv: {len(snapshot):,} rows")
snapshot.head(10)

## 10. `v_pool_drawdown.csv`

Filter to `BucketType='UnusedCapacity'`, group by `(Year, CommodityPoolID)`, pivot `MovementType`.

In [ ]:
pool_entries = ledger[ledger["BucketType"] == "UnusedCapacity"].copy()
pool_entries = pool_entries.dropna(subset=["Year", "CommodityPoolID"])
if len(pool_entries) > 0:
    pool_entries["Year"] = pool_entries["Year"].astype(int)
    pool_entries["CommodityPoolID"] = pool_entries["CommodityPoolID"].astype(int)

    drawdown = pool_entries.pivot_table(
        index=["Year", "CommodityPoolID", "CommodityShortName"],
        columns="MovementType",
        values="Quantity",
        aggfunc="sum",
        fill_value=0,
    ).reset_index()
    drawdown.columns.name = None
else:
    drawdown = pd.DataFrame(columns=["Year", "CommodityPoolID", "CommodityShortName"])
drawdown.to_csv(VIEWS / "v_pool_drawdown.csv", index=False)
print(f"v_pool_drawdown.csv: {len(drawdown):,} rows")
drawdown.head(10)

## 11. Validator — three checks

1. `SUM(Quantity) GROUP BY EventID = 0` for balanced MovementTypes.
2. `(SourceSystem, SourceID)` appears once or twice per event — not more.
3. Row count in the expected range.

In [ ]:
balanced_types = {"ALLOC", "ALLOCASSGN", "CONV", "CONVTRF", "TRF", "Banking", "LBA", "LBT"}
unbalanced_types = {"ECM"}

passes = []

balanced = ledger[ledger["MovementType"].isin(balanced_types)]
sums = balanced.groupby("EventID")["Quantity"].sum()
bad = sums[sums.abs() > 1e-6]
check1 = len(bad) == 0
passes.append(("balanced events sum to 0", check1, f"{len(bad)} events out of balance"))
if not check1:
    print("Out-of-balance events (first 10):")
    print(bad.head(10))

dupe = ledger[ledger["SourceSystem"] != "manual"].groupby(["SourceSystem", "SourceID"]).size()
weird = dupe[(dupe != 1) & (dupe != 2)]
check2 = len(weird) == 0
passes.append(("source_id count in {1,2}", check2, f"{len(weird)} keys with unexpected count"))

expected_low, expected_high = 2000, 4000
check3 = expected_low <= len(ledger) <= expected_high
passes.append((f"row count in [{expected_low}, {expected_high}]", check3, f"got {len(ledger):,}"))

print("Validator results:")
for name, ok, note in passes:
    sym = "OK" if ok else "FAIL"
    print(f"  [{sym}] {name} — {note}")
if not all(ok for _, ok, _ in passes):
    print("\n!! Validator flagged issues — inspect before trusting downstream views.")

## 12. Sanity check vs public 2023 report

Compare `v_cumulative_accounting` 2023 residential (SFRUU + MFRUU) to the numbers in
[thresholds.laketahoeinfo.org/CumulativeAccounting/Index/2023](https://thresholds.laketahoeinfo.org/CumulativeAccounting/Index/2023):

- Existing: **48,891**
- Banked: **512**
- Released allocations 2012–2023: **1,558**
- Remaining allocations: **960**
- RBUs remaining: **1,445**

Deltas are findings, not failures — the Feb-2024 Corral backup may be stale and the GIS FC reader isn't in this prototype yet.

In [ ]:
RESIDENTIAL_SHORT = ["SFRUU", "MFRUU"]
pub = {"Existing": 48891, "Banked": 512}

snap_2023 = snapshot[
    (snapshot["Year"] == 2023)
    & (snapshot["CommodityShortName"].isin(RESIDENTIAL_SHORT))
].copy()
print(f"2023 residential rows: {len(snap_2023)}")

bucket_cols = [c for c in ("Existing", "Banked", "Allocated", "UnusedCapacity", "BonusUnits") if c in snap_2023.columns]
totals = snap_2023[bucket_cols].sum(numeric_only=True) if bucket_cols else pd.Series(dtype=float)
print("\nComputed 2023 residential totals (SFRUU + MFRUU):")
print(totals.to_string())
print()
for bucket, expected in pub.items():
    got = float(totals.get(bucket, 0) or 0)
    delta = got - expected
    print(f"  {bucket:<18} got={got:>8.0f}  expected={expected:>8}  delta={delta:+.0f}")

print("\nNote: any large delta is a finding, not a failure — "
      "refine with the outstanding questions in erd/next_steps.md "
      "and the GIS FC reader (SDE plan Phase 3).")